<a href="https://colab.research.google.com/github/AnastasiyaOvechko/Digidrat/blob/main/PD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
from openpyxl import load_workbook

# ====================================================
# НАСТРОЙКИ
# ====================================================

FILE_SRC = "PD RMC 01.xlsx"
FILE_DST = "Pd_01 01 2026.xlsx"

SRC_INPUT_SHEET = "Аркуш1"
SRC_OUTPUT_SHEET = "Аркуш2"

DST_MAIN_SHEET = "PDm"
DST_BAL_SHEET = "бал нов"

DATE_LIMIT = pd.to_datetime("2024-06-01")

COMMENT_TEXT = "Обробка виконана автоматично згідно методички (п.19)."

# ====================================================
# ЧАСТЬ 1. PD RMC 01.xlsx
# ====================================================

# --- чтение Аркуш1 ---
df = pd.read_excel(
    FILE_SRC,
    sheet_name=SRC_INPUT_SHEET,
    header=2
)

# Fix: Rename 'Id заявки' to 'UID заявки' to match BASE_COLS if it exists
if 'Id заявки' in df.columns and 'UID заявки' not in df.columns:
    df.rename(columns={'Id заявки': 'UID заявки'}, inplace=True)

# Fix: Add 'Клиент' column to df if it's missing, with None values, to prevent KeyError
if 'Клиент' not in df.columns:
    df['Клиент'] = None

# удаляем МКК и клиент
if "Тип отчетности" in df.columns:
    df = df[~df["Тип отчетности"].isin(["Mkk", "Client"])]


# Дата установки статуса → без времени
STATUS_COL = "Дата установки статуса"
MOD_STATUS_COL = "Дата установки статуса_Модифікована"

df[STATUS_COL] = pd.to_datetime(df[STATUS_COL], errors="coerce")
df[MOD_STATUS_COL] = df[STATUS_COL].dt.date

# сортировка по модифицированной дате
df = df.sort_values(
    by=[
        "ОКПО",
        "Дата 1",
        "Дата установки статуса_Модифікована"
    ],
    ascending=[
        True,   # ОКПО — по возрастанию
        False,  # Дата 1 — от новой к старой
        False   # Дата установки статуса_Модифікована — от новой к старой
    ]
)


# проверка
if "ОКПО" in df.columns:
    _ = (
        df.sort_values(["ОКПО", MOD_STATUS_COL], ascending=[True, False])
        .groupby("ОКПО")[MOD_STATUS_COL]
        .apply(lambda x: x.is_monotonic_decreasing)
    )

# фильтр по Дата 1 и добавление столбцов
df["Дата 1"] = pd.to_datetime(df["Дата 1"], errors="coerce")
df = df[df["Дата 1"] >= DATE_LIMIT]

OKPO_COL = "ОКПО"
DATE1_COL = "Дата 1"

# оставляем ТОЛЬКО первую строку по ОКПО
df = df.drop_duplicates(
    subset=OKPO_COL,
    keep="first"
)

# добавляем два столбца после "Дата 1"
idx = df.columns.get_loc(DATE1_COL)
df.insert(idx + 1, "flag_1", 1)
df.insert(idx + 2, "flag_filter", 1)


# ---------- Аркуш2 ----------
with pd.ExcelWriter(
    FILE_SRC,
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:
    df.to_excel(writer, sheet_name=SRC_OUTPUT_SHEET, index=False)

wb = load_workbook(FILE_SRC)
ws = wb[SRC_OUTPUT_SHEET]
ws["S2"] = COMMENT_TEXT
wb.save(FILE_SRC)

# ====================================================
# ЧАСТЬ 2. Pd_01 01 2026.xlsx
# ====================================================


from openpyxl import load_workbook

wb = load_workbook(FILE_DST)
ws_pdm = wb[DST_MAIN_SHEET]
ws_bal = wb[DST_BAL_SHEET]

START_ROW = 5  # ДАННЫЕ НАЧИНАЮТСЯ С 5 СТРОКИ

for i, row in df.iterrows():
    r = START_ROW + i

    ws_pdm[f"C{r}"] = row["UID заявки"]
    ws_pdm[f"D{r}"] = row["Статус"]
    ws_pdm[f"E{r}"] = row["ОКПО"]
    ws_pdm[f"F{r}"] = row["Клиент"]
    ws_pdm[f"G{r}"] = row["Дата установки статуса_Модифікована"]
    ws_pdm[f"H{r}"] = row["Дата 1"]

    ws_bal[f"C{r}"] = row["ОКПО"]
    ws_bal[f"D{r}"] = row["Клиент"]
    ws_bal[f"E{r}"] = row["Дата 1"]
    ws_bal[f"F{r}"] = row["UID заявки"]

wb.save(FILE_DST)

print("✅ Обробка завершена.")


df_dst = pd.read_excel(FILE_DST, sheet_name=DST_MAIN_SHEET, header=4) # Specify header=4

# Rename columns to match BASE_COLS for consistency
if 'Uid заявки' in df_dst.columns:
    df_dst.rename(columns={'Uid заявки': 'UID заявки'}, inplace=True)
if 'Окпо' in df_dst.columns:
    df_dst.rename(columns={'Окпо': 'ОКПО'}, inplace=True)

# перенос ключевых данных
BASE_COLS = [
    "UID заявки",
    "Статус",
    "ОКПО",
    "Клиент",
    MOD_STATUS_COL,
    "Дата 1"
]

# Ensure all BASE_COLS are present in df_dst before assignment
for col in BASE_COLS:
    if col not in df_dst.columns:
        df_dst[col] = None # Add missing columns to df_dst with default None values

# Convert all column names to string type to prevent AttributeError with .lower()
df_dst.columns = df_dst.columns.map(str)

# Ensure 'Балл' column exists in df_dst before it's used in BAL_COLS
if 'Балл' not in df_dst.columns:
    df_dst['Балл'] = None


df_dst[BASE_COLS] = df[BASE_COLS].values

# qq1–qq12
qq_cols = [c for c in df.columns if c.lower().startswith("qq")]
# Ensure qq_cols from df are also in df_dst before assignment, creating if missing
for col in qq_cols:
    if col not in df_dst.columns:
        df_dst[col] = None
df_dst[qq_cols] = df[qq_cols].values

# sq → Yes / No → 1 / 0
sq_cols = [c for c in df_dst.columns if c.lower().startswith("sq")]
df_dst[sq_cols] = df_dst[sq_cols].replace({"Yes": 1, "No": 0})

# ---------- запись PDm ----------


BAL_COLS = ["ОКПО", "Клиент", "Дата 1", "UID заявки", "Балл"]
STOP_COLS = [c for c in df_dst.columns if c.lower().startswith("sq")]

df_bal = df_dst[BAL_COLS + STOP_COLS]



print("✅ Обробка завершена.")

✅ Обробка завершена.


ValueError: Length of values (5) does not match length of index (28)